# 🥉 Bronze Layer: Ingest Raw AAPL Stock Data
**Purpose:** Fetch daily stock data from the Yahoo Finance API, append a ticker symbol column for scalability, add auditing metadata, and append the raw records to our Bronze layer.

In [0]:
import yfinance as yf
import pandas as pd
from pyspark.sql import functions as F

In [0]:
# 1. Configuration
CATALOG = "portfolio"
SCHEMA = "market_data"
BRONZE_YF_TABLE = f"{CATALOG}.{SCHEMA}.bronze_stock_quotes"
TICKER = "AAPL"

# Create Catalog, Database if it doesn't exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG}.{SCHEMA}")

In [0]:
# 2. Fetch Data
## From YF
# Using '1d' for daily scheduled runs.
pdf_raw = yf.download(TICKER, period="1d")
pdf_raw.reset_index(inplace=True)

# Flatten columns if yfinance returns a MultiIndex
if isinstance(pdf_raw.columns, pd.MultiIndex):
    pdf_raw.columns = pdf_raw.columns.get_level_values(0)

# Convert Date to string for safe Spark transition
pdf_raw['Date'] = pdf_raw['Date'].dt.strftime('%Y-%m-%d')

In [0]:
pdf_raw

In [0]:
# 3. Convert to Spark and Add Metadata
## YF
df_bronze = spark.createDataFrame(pdf_raw)

df_bronze = df_bronze.withColumn("ticker_symbol", F.lit(TICKER)) \
                     .withColumn("ingestion_timestamp", F.current_timestamp()) \
                     .withColumn("source_system", F.lit("yfinance_api"))

# Alpha Vantage (JSON objects)
if not news_feed:
    print("No news found in the last 24 hours or API limit reached.")
else:
    # 5. Convert to Spark DataFrame and Add Metadata
    df_bronze_news = spark.createDataFrame(news_feed)

    df_bronze_news = df_bronze_news.withColumn("ticker_symbol", F.lit(TICKER)) \
                                   .withColumn("ingestion_timestamp", F.current_timestamp()) \
                                   .withColumn("source_system", F.lit("alphavantage_api"))

In [0]:
# 4. Write to Bronze (Append)
df_bronze.write.format("delta").mode("append").saveAsTable(BRONZE_YF_TABLE)

display(spark.read.table(BRONZE_YF_TABLE))